# Section 5: Monitoring & Observability
## AI-Native Software Architecture — O'Reilly Course

Traditional monitoring tells us whether the service is up.
LLM observability tells us whether the system behaved correctly.

We will add:
- tracing
- prompt/version tracking
- token/cost approximations
- deterministic evaluation
- LLM-as-a-judge rubric

In [ ]:
import support_utils
from support_utils import (
    primary_issue,
    escalation_response, blocked_response,
    FORBIDDEN_CLAIMS, parse_json_response,
)
from datetime import datetime, UTC
from typing import Any, Dict, List, Optional
import json

In [ ]:
# Toggle LLM mode without editing support_utils.py or restarting the kernel.
# Gemini (Vertex AI) is used when gemini_client is initialised in support_utils.
import support_utils
support_utils.USE_REAL_LLM = False   # set True to call a real LLM
print(f"USE_REAL_LLM = {support_utils.USE_REAL_LLM}")

## Step 5.1: Trace Events

A trace captures what happened during a request.
In production this could go to LangSmith, Langfuse, Arize, Datadog, or Braintrust.
For this notebook we use an in-memory list.

In [ ]:
TRACE_LOGS: List[Dict[str, Any]] = []

def estimate_tokens(text: str) -> int:
    return max(1, len(text) // 4)

def log_trace(request_id: str, stage: str, event: str, payload: Dict[str, Any]) -> None:
    TRACE_LOGS.append({
        "timestamp": datetime.now(UTC).isoformat(),
        "request_id": request_id,
        "stage": stage,
        "event": event,
        "payload": payload,
    })

def view_traces(request_id: Optional[str] = None) -> List[Dict[str, Any]]:
    rows = TRACE_LOGS
    if request_id:
        rows = [r for r in rows if r["request_id"] == request_id]
    return rows

def print_traces(request_id: Optional[str] = None) -> None:
    for r in view_traces(request_id):
        print(f"\n🧾 [{r['timestamp']}]")
        print(f"Request: {r['request_id']}")
        print(f"Stage: {r['stage']} | Event: {r['event']}")
        print(f"Payload: {r['payload']}")

log_trace("demo_1", "setup", "notebook_trace_initialized", {"status": "ok"})
print_traces()

## Step 5.2: Evaluation Metrics

We evaluate outputs using deterministic checks.

In production you may combine:
- deterministic validation
- reference-based checks
- LLM-as-a-judge
- human review
- product/business signals

In [ ]:
def evaluate_response(output: Dict[str, Any]) -> Dict[str, Any]:
    message = json.dumps(output).lower()
    return {
        "schema_present": isinstance(output, dict),
        "no_unsafe_refund_claim": not any(claim in message for claim in FORBIDDEN_CLAIMS),
        "has_next_action": bool(output.get("next_action") or output.get("message")),
        "requires_human_for_refund": (
            "refund" not in message
            or "human" in message
            or "escalat" in message
            or "verify" in message
        ),
    }

def score_eval(eval_result: Dict[str, bool]) -> float:
    values = list(eval_result.values())
    return sum(bool(v) for v in values) / len(values)

sample_eval = evaluate_response(escalation_response(primary_issue, "Refund requires human approval"))
print(sample_eval)
print("Score:", score_eval(sample_eval))

## Step 5.3: LLM-as-a-Judge Rubric

This rubric mirrors the slide exercise on evaluation dimensions:
- Faithfulness
- Conciseness
- Schema adherence

In [ ]:
JUDGE_RUBRIC = """
You are evaluating the output of a support assistant.

Score the response from 1-5 on:

1. Faithfulness:
Does the response stay grounded in the provided policy and avoid inventing actions?

2. Conciseness:
Is the response clear and direct without unnecessary detail?

3. Schema adherence:
Does the response follow the expected JSON or response structure?

Return JSON:
{
  "faithfulness": number,
  "conciseness": number,
  "schema_adherence": number,
  "reasoning": "short explanation"
}
"""

def llm_judge_prompt(output: Dict[str, Any], context: str) -> str:
    return f"""
{JUDGE_RUBRIC}

Context:
{context}

Assistant output:
{json.dumps(output, indent=2)}
"""

print(llm_judge_prompt(
    escalation_response(primary_issue, "Refund requires human approval"),
    "Duplicate charge refunds require human approval.",
))

## Section 5 takeaway

You cannot scale what you cannot measure.

Observability gives teams a way to:
- debug failures
- compare prompt versions
- catch regressions
- understand cost and latency
- improve the system over time

**Next:** Section 6 — composing everything into an end-to-end pipeline.